# Step 04: Interactive Hyperparameter Tuning & Sensitivity Analysis

Provides an interactive playground to experiment with training hyperparameters (Learning Rate, Epochs, Batch Size) and analyze their sensitivity on Fall Recall, Fall $F_1$-Score, and ROC-AUC.

### Key Experiments in this Notebook:
1. **Hyperparameter Selection**: Adjust `lr`, `epochs`, and `batch_size` in Cell 1 to train and evaluate your custom configuration.
2. **Learning Rate Sensitivity Sweep**: Automated sweep over $lr \in [10^{-4}, 5\cdot 10^{-4}, 10^{-3}, 5\cdot 10^{-3}]$ with visual comparison curves.
3. **Threshold Sensitivity & Ablation Analysis**: Evaluates performance gain of validation threshold tuning ($t^*$) vs. fixed $0.50$ cutoff.

In [1]:
# Step 1: Configuration & Hyperparameter Selection
%load_ext autoreload
%autoreload 2
import sys, importlib
sys.path.append("..") if ".." not in sys.path else None

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import src.config, src.dataset_utils, src.pipeline, src.trainer, src.representation_utils
importlib.reload(src.config)
importlib.reload(src.dataset_utils)
importlib.reload(src.pipeline)
importlib.reload(src.trainer)
importlib.reload(src.representation_utils)

from src.pipeline import run_model_training
from src.config import DATASET_CONFIGS

sns.set_theme(style="whitegrid")

# ==============================================================================
# EXPERIMENT CONFIGURATION
# ==============================================================================
dataset_key = "mmfall"   # "mmfall" | "mmwave" | "combined"
model_name = "rep3"      # "cnn" | "rep1" | "rep2" | "rep3"
rep_variant = "raw"      # Rep1: "raw"|"log_scaled"|"smooth" | Rep2: "raw"|"gaussian"|"trail" | Rep3: "raw"|"centered"|"normalized"

# Custom Hyperparameters:
epochs = 30              # Number of training epochs
lr = 1e-3                # Learning rate (Adam optimizer)
batch_size = 32          # Mini-batch size

print(f"Selected Dataset: {dataset_key.upper()}")
print(f"Selected Model Representation: {model_name.upper()} (Variant: '{rep_variant}')")
print(f"Hyperparameters: epochs={epochs}, lr={lr}, batch_size={batch_size}")

Selected Dataset: MMFALL
Selected Model Representation: REP3
Hyperparameters: epochs=30, lr=0.001, batch_size=32


In [2]:
# Step 2: Execute Training with Selected Hyperparameters
print(f"=== Running Experiment: model='{model_name}', variant='{rep_variant}', dataset='{dataset_key}', lr={lr}, epochs={epochs} ===")
acc, auc, cm = run_model_training(dataset_key, model_name=model_name, epochs=epochs, lr=lr, batch_size=batch_size, variant=rep_variant)

=== Running Experiment: model='rep3', dataset='mmfall', lr=0.001, epochs=30 ===

--- Training Representation_3_PointNet on mmFall (TI IWR1443) (epochs=30, lr=0.001, batch_size=32) ---
Epoch [01/30] Tr Loss: 0.5372, Acc: 76.23% | Val Loss: 0.3136, Acc: 98.81%, F1: 7.41%, Rec: 4.84% (Thresh: 0.70) [BEST SAVED]
Epoch [05/30] Tr Loss: 0.3353, Acc: 85.66% | Val Loss: 0.3593, Acc: 94.75%, F1: 8.56%, Rec: 25.00% (Thresh: 0.74) [BEST SAVED]
Epoch [10/30] Tr Loss: 0.3091, Acc: 86.48% | Val Loss: 0.2869, Acc: 96.38%, F1: 10.92%, Rec: 22.58% (Thresh: 0.72) [BEST SAVED]
Epoch [15/30] Tr Loss: 0.2515, Acc: 89.26% | Val Loss: 0.3597, Acc: 95.73%, F1: 6.92%, Rec: 16.13% (Thresh: 0.84)
Epoch [20/30] Tr Loss: 0.2494, Acc: 87.79% | Val Loss: 0.1646, Acc: 98.04%, F1: 14.53%, Rec: 16.94% (Thresh: 0.88) [BEST SAVED]
Epoch [25/30] Tr Loss: 0.2062, Acc: 90.90% | Val Loss: 0.3115, Acc: 96.67%, F1: 10.26%, Rec: 19.35% (Thresh: 0.88)
Epoch [30/30] Tr Loss: 0.2366, Acc: 90.33% | Val Loss: 0.2874, Acc: 95.58%, F1

In [4]:
# Visual Evaluation (Confusion Matrix, Metrics Breakdown & Timestamped Parameters)
from src.config import DATASET_CONFIGS, BENCHMARKS_DIR
tag_alias = "mmfall" if dataset_key == "mmfall" else ("ti" if dataset_key == "mmwave" else "combined")
res_json_path = BENCHMARKS_DIR / f"benchmark_results_{tag_alias}.json"
dataset_name = DATASET_CONFIGS[dataset_key].name

if res_json_path.exists():
    bm_res = json.load(open(res_json_path))
    rep_key_map = {
        "cnn": "Radar4DCNN_Baseline",
        "rep1": f"Representation_1_Spectrogram_{rep_variant}",
        "rep2": f"Representation_2_Projections_{rep_variant}",
        "rep3": f"Representation_3_PointNet_{rep_variant}"
    }
    rep_key = rep_key_map.get(model_name, "Radar4DCNN_Baseline")
    if rep_key not in bm_res:
        # Fallback search if exact key variant missing
        matching_keys = [k for k in bm_res.keys() if model_name.lower() in k.lower()]
        rep_key = matching_keys[-1] if len(matching_keys) > 0 else list(bm_res.keys())[-1]
    
    if rep_key in bm_res:
        res_data = bm_res[rep_key]
        cm_matrix = np.array(res_data["confusion_matrix"])
        ts = res_data.get("timestamp", "N/A")
        run_id = res_data.get("run_id", f"benchmark_{rep_key}")
        hparams = res_data.get("hyperparameters", {})
        ep_info = hparams.get("epochs", 15)
        lr_info = hparams.get("lr", 1e-3)
        bs_info = hparams.get("batch_size", 32)
        th_info = res_data.get("optimal_threshold", 0.50)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), dpi=100)
        
        # 1. Confusion Matrix
        sns.heatmap(cm_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
                    xticklabels=["Pred ADL (0)", "Pred Fall (1)"],
                    yticklabels=["Actual ADL (0)", "Actual Fall (1)"])
        axes[0].set_title(f"Confusion Matrix ({rep_key} | {dataset_name})", fontweight="bold", fontsize=11)
        
        tn, fp, fn, tp = cm_matrix.ravel() if cm_matrix.size == 4 else (0,0,0,0)
        axes[0].text(0.5, 0.2, f"TN: {tn}", ha="center", va="center", color="navy", fontsize=11, fontweight="bold")
        axes[0].text(1.5, 0.2, f"FP (False Alarm): {fp}", ha="center", va="center", color="darkred", fontsize=11, fontweight="bold")
        axes[0].text(0.5, 1.2, f"FN (MISSED FALL): {fn}", ha="center", va="center", color="darkred", fontsize=11, fontweight="bold")
        axes[0].text(1.5, 1.2, f"TP (CAUGHT): {tp}", ha="center", va="center", color="darkgreen", fontsize=11, fontweight="bold")
        
        # 2. Performance Metrics
        metrics = ["Accuracy", "Fall Recall", "Fall Precision", "Fall F1-Score", "ROC-AUC"]
        vals = [res_data["accuracy"], res_data["recall"], res_data["precision"], res_data["f1_score"], res_data["roc_auc"]]
        colors = ["#3498db", "#2ecc71", "#9b59b6", "#e74c3c", "#f39c12"]
        
        bars = axes[1].bar(metrics, [v*100 for v in vals], color=colors, edgecolor="black", width=0.55)
        axes[1].set_ylim(0, 115)
        axes[1].set_ylabel("Percentage (%) / Score")
        axes[1].set_title(f"Performance Summary ({rep_key} | {dataset_name})", fontweight="bold", fontsize=11)
        axes[1].set_xticklabels(metrics, rotation=15, ha="right")
        
        for bar in bars:
            h = bar.get_height()
            axes[1].text(bar.get_x() + bar.get_width()/2.0, h + 2, f"{h:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=10)
            
        title_text = f"Model Evaluation Summary: Model='{model_name.upper()}' ({rep_key}) | Dataset='{dataset_key.upper()}' ({dataset_name})\nParameters: epochs={ep_info}, lr={lr_info}, batch_size={bs_info}, $t^*={th_info:.2f}$ | Run Date: {ts}"
        plt.suptitle(title_text, fontweight="bold", fontsize=12, y=1.04)
        plt.tight_layout()
        
        # Save exact matching PNG figure alongside JSON run file
        run_png_path = BENCHMARKS_DIR / f"{run_id}.png"
        plt.savefig(run_png_path, bbox_inches="tight", dpi=300)
        print(f"Saved matching benchmark plot image to: {run_png_path}")
        plt.show()
else:
    print(f"No benchmark results found yet at {res_json_path}. Run training step first!")

No benchmark results found yet at C:\Users\joeyw\GitProjects\mmWaves-Fall-Detection\models\benchmarks\benchmark_results_mmfall.json. Run training step first!
